# Cilindro

**Módulo 06 — Geometria Espacial** · Notebook 11 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0611a_cilindro.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0608a_prismas_e_paralelepipedos.ipynb`](0608a_prismas_e_paralelepipedos.ipynb) e [`0609a_volume_e_principio_de_cavalieri.ipynb`](0609a_volume_e_principio_de_cavalieri.ipynb)
- Módulo 04 — Geometria Plana: [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (comprimento da circunferência) e [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (área do círculo)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- reconhecer superfícies cilíndricas e os elementos de um cilindro;
- classificar cilindros e identificar suas seções (transversal e meridiana);
- deduzir e usar as fórmulas das áreas lateral e total do cilindro reto;
- deduzir e usar a fórmula do volume do cilindro com o Princípio de Cavalieri.

**Tempo estimado:** 70 a 90 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [ ]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets
import math
import numpy as np

## Qual é a lata mais econômica?

Uma fábrica quer produzir latas cilíndricas que guardem exatamente **350 mL** (350 cm³). Ela pode fazer latas finas e altas, ou largas e baixas. Todas guardam a mesma quantidade, mas gastam quantidades diferentes de alumínio.

Qual formato gasta **menos** alumínio?

🕹️ No widget abaixo, você escolhe o raio da lata; a altura é ajustada para que o volume continue sendo 350 cm³. O gráfico mostra a área total de alumínio para cada raio. (As fórmulas usadas aqui serão explicadas nas seções 3 e 4. Por enquanto, confie nelas e explore!)

In [2]:
def explorar_lata(raio: float) -> None:
    """Desenha a lata de 350 cm³ com o raio escolhido e marca sua área no gráfico."""
    volume = 350
    altura = volume / (math.pi * raio**2)
    area_total = 2 * math.pi * raio * altura + 2 * math.pi * raio**2

    fig = u3d.criar_figura(titulo="A lata de 350 cm³")
    fig = u3d.desenhar_cilindro(fig, (0, 0, 0), raio, (0, 0, altura))
    u3d.mostrar(fig)

    # A curva do gráfico: repetimos a mesma conta para vários raios, de 1,5 a 8.
    raios = []
    areas = []
    for k in range(66):
        r = 1.5 + k * 0.1
        h = volume / (math.pi * r**2)
        raios.append(r)
        areas.append(2 * math.pi * r * h + 2 * math.pi * r**2)

    grafico = u3d.criar_grafico(
        titulo="Alumínio gasto em cada lata",
        rotulo_x="raio (cm)",
        rotulo_y="área total (cm²)",
    )
    grafico = u3d.desenhar_curva(grafico, raios, areas, rotulo="área total")
    grafico = u3d.marcar_ponto_no_grafico(
        grafico, raio, area_total, rotulo=f"r = {raio:.1f}"
    )
    u3d.mostrar(grafico)

    print(f"Altura: {altura:.2f} cm")
    print(f"Área total: {area_total:.2f} cm²")
    print(f"Razão altura / raio: {altura / raio:.2f}")


interact(
    explorar_lata,
    raio=widgets.FloatSlider(
        min=1.5, max=8, step=0.1, value=3.3, description="raio (cm)",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=3.3, continuous_update=False, description='raio (cm)', max=8.0, min=1.…

O gráfico tem um ponto mais baixo: perto de $r \approx 3{,}8$ cm, a lata gasta o mínimo de alumínio. E repare na razão $\frac{h}{r}$ nesse ponto: ela fica perto de **2**. A lata mais econômica tem a **altura igual ao diâmetro**!

As latas de verdade costumam ser mais altas que isso. Os fabricantes levam em conta outras coisas além da área: a tampa e o fundo são mais grossos (e mais caros) que a lateral, a lata precisa caber na mão e nas geladeiras, e o formato também é uma questão de marca.

Neste notebook você vai entender o cilindro do zero: de onde vêm as fórmulas de área e volume e por que esse cilindro "especial", com altura igual ao diâmetro, tem nome próprio.

## 1. Superfícies cilíndricas

Imagine um segmento de reta que se move, sempre **paralelo** a uma direção fixa, apoiado numa curva. O "rastro" que ele deixa é uma **superfície cilíndrica**.

> 📌 **Definição.**
> - A curva em que o segmento se apoia é a **diretriz**.
> - Cada posição do segmento é uma **geratriz**.
> - Quando a diretriz é uma **circunferência**, a superfície é **cilíndrica circular**.

A diretriz não precisa ser uma circunferência:
- uma **telha ondulada** é uma superfície cilíndrica cuja diretriz é uma curva em forma de onda;
- uma **calha** de chuva tem diretriz em forma de "U";
- um **tubo** ou um cano tem diretriz circular.

🕹️ Escolha a diretriz e acompanhe a geratriz "varrendo" a superfície.

In [3]:
def explorar_superficie(diretriz: str, percorrido: float, inclinacao: float) -> None:
    """Mostra a geratriz varrendo a superfície cilíndrica a partir da diretriz."""
    # Cada diretriz é uma lista de pontos no plano z = 0.
    if diretriz == "Circunferência (tubo)":
        angulos = np.linspace(0, 2 * np.pi, 81)
        curva = [(2 * np.cos(t), 2 * np.sin(t), 0) for t in angulos]
    elif diretriz == "Onda (telha ondulada)":
        curva = [(x, 0.3 * np.sin(3 * x), 0) for x in np.linspace(-3, 3, 121)]
    else:
        # O fundo da calha: uma semicircunferência de raio 1.
        curva = [(np.cos(t), np.sin(t), 0) for t in np.linspace(np.pi, 2 * np.pi, 61)]

    # Seno e cosseno decompõem o segmento de comprimento 4, inclinado de θ em relação
    # ao plano da diretriz, numa parte horizontal e numa parte vertical.
    theta = math.radians(inclinacao)
    deslocamento = (4 * math.cos(theta), 0, 4 * math.sin(theta))

    # Só a parte da curva que a geratriz já percorreu.
    quantidade = max(2, round(percorrido * len(curva)))
    trecho = curva[:quantidade]
    x, y, z = trecho[-1]
    fim_da_geratriz = (x + deslocamento[0], y + deslocamento[1], z + deslocamento[2])

    fig = u3d.criar_figura(titulo="A geratriz varrendo a superfície")
    fig = u3d.desenhar_superficie_cilindrica(fig, trecho, deslocamento)
    fig = u3d.desenhar_curva_no_espaco(
        fig, curva, rotulo="diretriz", cor=u3d.CORES["destaque"]
    )
    fig = u3d.desenhar_segmento(
        fig, trecho[-1], fim_da_geratriz, rotulo="geratriz", cor=u3d.CORES["reta"],
        espessura=8,
    )
    u3d.mostrar(fig)


interact(
    explorar_superficie,
    diretriz=widgets.Dropdown(
        options=[
            "Circunferência (tubo)", "Onda (telha ondulada)", "U (calha)",
        ],
        value="Circunferência (tubo)",
        description="diretriz",
    ),
    percorrido=widgets.FloatSlider(
        min=0.1, max=1, step=0.1, value=0.5, description="percorrido",
        continuous_update=False,
    ),
    inclinacao=widgets.IntSlider(
        min=60, max=90, step=10, value=90, description="inclinação (°)",
        continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='diretriz', options=('Circunferência (tubo)', 'Onda (telha ondulada…

**Pare e pense**

1. Na telha ondulada, qual é a diretriz e quais são as geratrizes?
2. Uma superfície cilíndrica pode ter diretriz quadrada? Que objeto teria essa forma?

<details>
<summary>Ver respostas</summary>

1. A diretriz é a curva em onda (o perfil da telha, visto de frente). As geratrizes são os segmentos retos ao longo do comprimento da telha.
2. Pode: é a superfície lateral de um prisma de base quadrada (uma caixa sem tampa e sem fundo, como um duto de ar-condicionado).
</details>

## 2. Cilindro circular

> 📌 **Definição.** Considere um círculo de centro $O$ e raio $r$ num plano $\alpha$, e um plano $\beta$ paralelo a $\alpha$. O **cilindro circular** é o sólido formado por todos os segmentos paralelos a uma direção dada, com uma extremidade no círculo e a outra em $\beta$.

**Elementos**
- **Bases:** dois círculos congruentes, de raio $r$, em planos paralelos.
- **Eixo:** o segmento $OO'$ que liga os centros das bases.
- **Geratrizes:** os segmentos paralelos ao eixo, com extremidades nas circunferências das bases; o comprimento delas é $g$.
- **Altura** $h$: a distância entre os planos das bases (notebook 04).

### Classificação

- **Cilindro reto:** as geratrizes são perpendiculares às bases. Então $g = h$. O cilindro reto também é chamado **cilindro de revolução**, porque é gerado por um **retângulo girando** em torno de um de seus lados.
- **Cilindro oblíquo:** as geratrizes são oblíquas às bases. Se a geratriz forma ângulo $\theta$ com o plano da base, $h = g \sin\theta$.

> ⚠️ **Erro comum.** No cilindro oblíquo, usar a geratriz como altura. A altura é a distância entre os planos das bases, medida na perpendicular; a geratriz é inclinada e é maior que $h$.

### Seções

- **Seção transversal:** feita por um plano paralelo às bases. É sempre um **círculo congruente** à base.
- **Seção meridiana:** feita por um plano que contém o **eixo**. No cilindro reto, é um **retângulo** $2r \times h$; no oblíquo, um **paralelogramo**.

> 📌 **Definição.** Um **cilindro equilátero** é um cilindro reto cuja seção meridiana é um **quadrado**: $h = 2r$. É exatamente a "lata ótima" do gancho!

> 💡 **Dica.** Se você cortar um salame ou um cabo de vassoura com um corte **inclinado**, a seção não é um círculo: é uma **elipse**, uma "circunferência esticada". Você verá mais sobre elipses no módulo 07.

🕹️ Monte um cilindro e escolha o tipo de corte.

In [ ]:
def explorar_cilindro(
    raio: float, geratriz: float, inclinacao: int, giro_da_secao: int, secao: str
) -> None:
    """Desenha um cilindro, classifica-o e mostra a seção escolhida."""
    theta = math.radians(inclinacao)
    altura = geratriz * math.sin(math.radians(inclinacao))
    deslocamento = (geratriz * math.cos(theta), 0, altura)
    meio_do_eixo = (deslocamento[0] / 2, 0, altura / 2)

    fig = u3d.criar_figura(titulo="O cilindro e suas seções")
    fig = u3d.desenhar_cilindro(
        fig, (0, 0, 0), raio, deslocamento, opacidade=0.3, mostrar_eixo=True,
        mostrar_geratrizes=8,
    )

    avisos = []
    if secao == "Transversal":
        circulo = u3d.poligono_regular(60, raio=raio, centro=meio_do_eixo)
        fig = u3d.desenhar_poligono(
            fig, circulo, cor=u3d.CORES["destaque"], opacidade=0.6
        )
    elif secao == "Meridiana":
        quadrilatero = u3d.secao_meridiana_do_cilindro(
            (0, 0, 0), raio, deslocamento, giro=giro_da_secao
        )
        fig = u3d.desenhar_poligono(
            fig, quadrilatero, cor=u3d.CORES["destaque"], opacidade=0.6
        )
        A, B, C, D = quadrilatero
        lado1 = u3d.medir_distancia(A, B)
        lado2 = u3d.medir_distancia(B, C)
        # Um paralelogramo com as diagonais iguais é um retângulo.
        if abs(u3d.medir_distancia(A, C) - u3d.medir_distancia(B, D)) > 1e-9:
            nome = "paralelogramo"
        elif abs(lado1 - lado2) > 1e-9:
            nome = "retângulo"
        else:
            nome = "quadrado"
        avisos.append(f"Seção meridiana: {nome} de lados {lado1:.2f} e {lado2:.2f}")
        if inclinacao < 90 and nome != "paralelogramo":
            avisos.append(
                f"Neste giro o diâmetro forma 90° com a geratriz: um {nome} também é "
                "um paralelogramo, só que com ângulos retos."
            )
    elif secao == "Inclinada (só no cilindro reto)":
        if inclinacao == 90:
            # Um plano pelo meio do eixo, inclinado 30° em relação à horizontal.
            normal = (math.sin(math.radians(30)), 0, math.cos(math.radians(30)))
            elipse = u3d.secao_plana_do_cilindro_reto(
                raio, altura, (meio_do_eixo, normal)
            )
            if elipse is None:
                avisos.append(
                    "Neste cilindro, o plano inclinado sai pelas bases e a seção não "
                    "fecha: aumente a geratriz ou diminua o raio."
                )
            else:
                fig = u3d.desenhar_poligono(
                    fig, elipse, cor=u3d.CORES["destaque"], opacidade=0.6
                )
        else:
            avisos.append(
                "Escolha inclinação 90° (cilindro reto) para ver o corte inclinado."
            )
    u3d.mostrar(fig)

    if inclinacao < 90:
        classificacao = "oblíquo"
    elif abs(altura - 2 * raio) < 1e-9:
        classificacao = "reto e equilátero"
    else:
        classificacao = "reto"
    print(f"Classificação: {classificacao}")
    print(f"Geratriz: {geratriz:.2f}; altura: {altura:.2f}")
    for aviso in avisos:
        print(aviso)


interact(
    explorar_cilindro,
    raio=widgets.FloatSlider(
        min=0.5, max=3, step=0.25, value=2, description="raio",
        continuous_update=False,
    ),
    geratriz=widgets.FloatSlider(
        min=1, max=8, step=0.5, value=5, description="geratriz",
        continuous_update=False,
    ),
    inclinacao=widgets.IntSlider(
        min=40, max=90, step=10, value=90, description="inclinação (°)",
        continuous_update=False,
    ),
    giro_da_secao=widgets.IntSlider(
        min=0, max=180, step=15, value=0, description="giro da seção (°)",
        continuous_update=False,
    ),
    secao=widgets.Dropdown(
        options=[
            "Nenhuma", "Transversal", "Meridiana", "Inclinada (só no cilindro reto)",
        ],
        value="Nenhuma",
        description="seção",
    ),
);

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='raio', max=3.0, min=0.5, st…

**Pare e pense**

1. A seção meridiana de um cilindro reto é um quadrado de lado 6. Quais são o raio e a altura?
2. Num cilindro oblíquo, a geratriz mede 10 e forma 60° com a base. Qual é a altura?

<details>
<summary>Ver respostas</summary>

1. $2r = 6$ e $h = 6$, então $r = 3$ e $h = 6$. É um cilindro equilátero.
2. $h = 10 \cdot \sin 60° = 5\sqrt{3} \approx 8{,}66$.
</details>

## 3. Planificação e áreas

### A planificação do cilindro reto

Corte a superfície lateral de um cilindro reto ao longo de uma geratriz e desenrole: ela vira um **retângulo**.
- A altura do retângulo é a altura $h$ do cilindro.
- O comprimento do retângulo é o comprimento da circunferência da base: $2\pi r$.

As duas bases são círculos de raio $r$.

🕹️ Desenrole e enrole o cilindro no widget abaixo.

In [5]:
def explorar_enrolamento(raio: float, altura: float, enrolamento: float) -> None:
    """Enrola o retângulo planificado até formar o cilindro."""
    fig = u3d.criar_figura(titulo="Do retângulo ao cilindro")
    fig = u3d.desenhar_enrolamento_do_cilindro(fig, raio, altura, enrolamento)
    if enrolamento == 0:
        # De frente para o plano do retângulo, ele aparece em verdadeira grandeza.
        fig = u3d.definir_camera(fig, olho=(0, -2.2, 0.2))
    else:
        fig = u3d.definir_camera(fig, olho=(0.8, -1.8, 0.8))
    u3d.mostrar(fig)

    print(f"Retângulo: {2 * math.pi * raio:.2f} × {altura}")


interact(
    explorar_enrolamento,
    raio=widgets.FloatSlider(
        min=0.5, max=3, step=0.25, value=1.5, description="raio",
        continuous_update=False,
    ),
    altura=widgets.FloatSlider(
        min=1, max=6, step=0.5, value=4, description="altura",
        continuous_update=False,
    ),
    enrolamento=widgets.FloatSlider(
        min=0, max=1, step=0.1, value=0, description="enrolamento",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=1.5, continuous_update=False, description='raio', max=3.0, min=0.5, st…

### As fórmulas das áreas (cilindro reto)

$$S_\ell = 2\pi r h \qquad S_b = \pi r^2 \qquad S_t = S_\ell + 2S_b = 2\pi r h + 2\pi r^2 = 2\pi r(h + r)$$

No **cilindro equilátero** ($h = 2r$): $S_\ell = 4\pi r^2$ e $S_t = 6\pi r^2$.

### De onde vem o $2\pi r h$? A ponte com o prisma

Inscreva no cilindro um prisma regular de $n$ lados (as bases do prisma são polígonos regulares inscritos nas bases do cilindro). A área lateral do prisma é $2p \cdot h$ (notebook 08). Quando $n$ cresce, o perímetro $2p$ do polígono se aproxima do comprimento da circunferência, $2\pi r$, e a área lateral do prisma se aproxima de $2\pi r h$.

> 💡 **Dica.** No cilindro **oblíquo**, a área lateral também é "perímetro da seção reta × geratriz", como no prisma oblíquo. Mas a seção reta de um cilindro oblíquo é uma elipse, cujo perímetro não tem uma fórmula simples. Por isso, as fórmulas de área deste notebook são para o cilindro **reto**.

**Aplicações:** o rótulo de uma lata (área lateral), a tinta para pintar uma caixa d'água cilíndrica, o papel para embrulhar um canudo ou um tubo.

🕹️ Veja os prismas "virando" cilindro.

In [6]:
def explorar_prismas(lados: int) -> None:
    """Inscreve um prisma regular no cilindro de raio 2 e altura 5; compara as áreas."""
    fig = u3d.criar_figura(titulo="Um prisma regular inscrito no cilindro")
    fig = u3d.desenhar_cilindro(fig, (0, 0, 0), 2, (0, 0, 5), opacidade=0.15)
    prisma_inscrito = u3d.prisma(u3d.poligono_regular(lados, raio=2), (0, 0, 5))
    fig = u3d.desenhar_poliedro(fig, *prisma_inscrito, opacidade=0.6)
    u3d.mostrar(fig)

    lado_do_poligono = 2 * 2 * math.sin(math.pi / lados)
    perimetro = lados * lado_do_poligono
    area_lateral_prisma = perimetro * 5
    area_lateral_cilindro = 2 * math.pi * 2 * 5
    print(f"Área lateral do prisma: {area_lateral_prisma:.4f}")
    print(f"Área lateral do cilindro: {area_lateral_cilindro:.4f}")
    print(f"Diferença: {area_lateral_cilindro - area_lateral_prisma:.4f}")

    # A mesma conta para vários valores de n: a tabela se aproxima de 20π ≈ 62,83.
    print("\n   n | área lateral do prisma")
    for n in [3, 6, 12, 24, 48, 96, 1000]:
        area_lateral = n * (2 * 2 * math.sin(math.pi / n)) * 5
        print(f"{n:>4} | {area_lateral:.4f}")


interact(
    explorar_prismas,
    lados=widgets.IntSlider(
        min=3, max=60, step=1, value=6, description="lados",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=6, continuous_update=False, description='lados', max=60, min=3), Output(…

**Pare e pense**

1. Calcule as áreas lateral e total de um cilindro reto de raio 3 e altura 10 (deixe em função de $\pi$).
2. Um rótulo cobre toda a lateral de uma lata de raio 3,3 cm e altura 12 cm. Qual é a área do rótulo?

<details>
<summary>Ver respostas</summary>

1. $S_\ell = 2\pi \cdot 3 \cdot 10 = 60\pi$; $S_t = 60\pi + 2 \cdot 9\pi = 78\pi$.
2. $2\pi \cdot 3{,}3 \cdot 12 \approx 248{,}8$ cm².
</details>

## 4. Volume do cilindro

**Teorema**
> O volume de um cilindro (reto ou oblíquo) é o produto da área da base pela altura:
> $$\boxed{\text{volume} = \pi r^2 \cdot h}$$

**Demonstração (com Cavalieri).** Coloque o cilindro e um prisma com base de mesma área ($\pi r^2$) e mesma altura $h$ apoiados num mesmo plano. Todo plano paralelo às bases corta o cilindro num círculo congruente à base (área $\pi r^2$) e o prisma num polígono congruente à base do prisma (também área $\pi r^2$). Pelo Princípio de Cavalieri (notebook 09), os volumes são iguais. O volume do prisma é $S_b \cdot h = \pi r^2 h$. ∎

No **cilindro equilátero** ($h = 2r$): volume $= 2\pi r^3$.

**Aplicações:** caixas d'água cilíndricas, latas, canos e mangueiras (a água "parada" dentro deles), poços, silos de grãos.

> 💡 **Dica.** Para converter o volume em litros, lembre: $1 \text{ dm}^3 = 1$ L e $1 \text{ m}^3 = 1000$ L (notebook 09).

🕹️ Compare as seções do cilindro e de um prisma de base quadrada com a mesma área.

In [7]:
def comparar_cilindro_e_prisma(
    raio: float, altura: float, inclinacao: int, altura_do_corte: float
) -> None:
    """Corta um cilindro oblíquo e um prisma de mesma área da base pelo mesmo plano."""
    area_base = math.pi * raio**2
    lado_do_quadrado = math.sqrt(area_base)
    volume_cilindro = area_base * altura
    volume_prisma = lado_do_quadrado**2 * altura

    # A geratriz é h / sen θ, para que a altura do cilindro seja a escolhida.
    theta = math.radians(inclinacao)
    geratriz = altura / math.sin(theta)
    deslocamento = (geratriz * math.cos(theta), 0, altura)

    # O prisma fica à esquerda do cilindro, sem encostar nele.
    raio_do_quadrado = lado_do_quadrado / math.sqrt(2)
    x_do_prisma = -(raio + raio_do_quadrado + 1)
    base_do_prisma = u3d.poligono_regular(
        4, lado=lado_do_quadrado, centro=(x_do_prisma, 0, 0)
    )

    # O centro do círculo da seção fica sobre o eixo, na altura do corte.
    z_do_corte = altura_do_corte * altura
    x_do_circulo = altura_do_corte * deslocamento[0]
    circulo = u3d.poligono_regular(60, raio=raio, centro=(x_do_circulo, 0, z_do_corte))
    quadrado = u3d.poligono_regular(
        4, lado=lado_do_quadrado, centro=(x_do_prisma, 0, z_do_corte)
    )

    fig = u3d.criar_figura(titulo="Mesma altura, mesma área da seção: mesmo volume")
    fig = u3d.desenhar_cilindro(fig, (0, 0, 0), raio, deslocamento, opacidade=0.25)
    fig = u3d.desenhar_poliedro(
        fig, *u3d.prisma(base_do_prisma, (0, 0, altura)), opacidade=0.25
    )
    fig = u3d.desenhar_poligono(
        fig, circulo, cor=u3d.CORES["destaque"], opacidade=0.7
    )
    fig = u3d.desenhar_poligono(
        fig, quadrado, cor=u3d.CORES["destaque"], opacidade=0.7
    )
    # O plano de corte precisa ser largo o bastante para alcançar os dois sólidos.
    esquerda = x_do_prisma - raio_do_quadrado
    direita = x_do_circulo + raio
    fig = u3d.desenhar_plano(
        fig, ((esquerda + direita) / 2, 0, z_do_corte), (0, 0, 1), opacidade=0.1,
        tamanho=direita - esquerda + 2,
    )
    u3d.mostrar(fig)

    print(f"Área da seção do cilindro: π · {raio}² = {area_base:.2f}")
    area_quadrado = lado_do_quadrado**2
    print(f"Área da seção do prisma: {lado_do_quadrado:.2f}² = {area_quadrado:.2f}")
    print(f"Volume do cilindro: {volume_cilindro:.2f}")
    print(f"Volume do prisma: {volume_prisma:.2f}")


interact(
    comparar_cilindro_e_prisma,
    raio=widgets.FloatSlider(
        min=0.5, max=2.5, step=0.25, value=1.5, description="raio",
        continuous_update=False,
    ),
    altura=widgets.FloatSlider(
        min=1, max=6, step=0.5, value=4, description="altura",
        continuous_update=False,
    ),
    inclinacao=widgets.IntSlider(
        min=50, max=90, step=10, value=70, description="inclinação (°)",
        continuous_update=False,
    ),
    altura_do_corte=widgets.FloatSlider(
        min=0, max=1, step=0.1, value=0.5, description="altura do corte",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=1.5, continuous_update=False, description='raio', max=2.5, min=0.5, st…

**Pare e pense**

1. Qual é o volume de um cilindro reto de raio 3 e altura 10?
2. Se o raio de um cilindro dobra e a altura continua a mesma, o que acontece com o volume?

<details>
<summary>Ver respostas</summary>

1. $\pi \cdot 9 \cdot 10 = 90\pi \approx 282{,}7$.
2. Fica multiplicado por 4, porque o volume depende de $r^2$.
</details>

## Resumo

**Superfície cilíndrica:** gerada por uma reta (geratriz) que se move paralela a uma direção, apoiada numa curva (diretriz).

**Cilindro circular:** bases circulares congruentes (raio $r$) em planos paralelos; eixo $OO'$; geratrizes $g$; altura $h$.
- **Reto** (de revolução): $g = h$; gerado por um retângulo girando em torno de um lado.
- **Oblíquo:** $h = g \sin\theta$.
- **Equilátero:** reto com $h = 2r$ (seção meridiana quadrada).

**Seções:** transversal = círculo congruente à base; meridiana = retângulo $2r \times h$ (reto) ou paralelogramo (oblíquo); inclinada (reto) = elipse.

**Áreas (cilindro reto):** $S_\ell = 2\pi r h$; $S_b = \pi r^2$; $S_t = 2\pi r(h + r)$. Equilátero: $S_t = 6\pi r^2$.

**Volume (reto ou oblíquo):** $\pi r^2 h$. Equilátero: $2\pi r^3$.

**Lata ótima:** para um volume fixo, a menor área total aparece quando $h = 2r$ (cilindro equilátero).

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**. Use `math.pi` nos cálculos.

O gabarito comentado está em [`0611b_cilindro_gabarito.ipynb`](0611b_cilindro_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) A seção meridiana de um cilindro reto é um retângulo.
# (b) No cilindro equilátero, a altura é igual ao raio.
# (c) Num cilindro oblíquo, a altura é menor que a geratriz.
# (d) Um cilindro reto e um oblíquo, com o mesmo raio e a mesma altura, têm o mesmo
#     volume.
# (e) A planificação da superfície lateral de um cilindro reto é um retângulo.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": True, "b": False, "c": True, "d": True, "e": True}
dicas_ex1 = {
    "a": "a definição de seção meridiana (seção 2 - Cilindro circular)",
    "b": "a definição de cilindro equilátero (seção 2 - Cilindro circular)",
    "c": "h = g sen θ, e sen θ < 1 (seção 2 - Cilindro circular)",
    "d": "a demonstração com Cavalieri (seção 4 - Volume do cilindro)",
    "e": "a superfície lateral desenrolada (seção 3 - Planificação e áreas)",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; {dicas_ex1[item]}."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: um cilindro reto tem raio 3 e altura 10. Calcule, em Python:
# (a) a área lateral;
# (b) a área total;
# (c) o volume.
area_lateral_ex2 = ...
area_total_ex2 = ...
volume_ex2 = ...

print(area_lateral_ex2, area_total_ex2, volume_ex2)

In [ ]:
# Verificação
respostas_ex2 = [
    ("a", area_lateral_ex2, 60 * math.pi, "S_ℓ = 2πrh",
     "seção 3 - Planificação e áreas"),
    ("b", area_total_ex2, 78 * math.pi, "S_t = 2πr(h + r)",
     "seção 3 - Planificação e áreas"),
    ("c", volume_ex2, 90 * math.pi, "volume = πr²h", "seção 4 - Volume do cilindro"),
]
for item, resposta, certa, dica, secao in respostas_ex2:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), {dica} ({secao})."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: um cilindro equilátero tem raio 2. Calcule, em Python:
# (a) a altura;
# (b) a área total;
# (c) o volume.
altura_ex3 = ...
area_total_ex3 = ...
volume_ex3 = ...

print(altura_ex3, area_total_ex3, volume_ex3)

In [ ]:
# Verificação
respostas_ex3 = [
    ("a", altura_ex3, 4, "seção 2 - Cilindro circular"),
    ("b", area_total_ex3, 24 * math.pi, "seção 3 - Planificação e áreas"),
    ("c", volume_ex3, 16 * math.pi, "seção 4 - Volume do cilindro"),
]
for item, resposta, certa, secao in respostas_ex3:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), lembre que no cilindro equilátero "
        f"h = 2r ({secao})."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: a caixa d'água. Uma caixa d'água cilíndrica tem 1,2 m de
# diâmetro e 1,5 m de altura. Quantos litros ela comporta?
litros_ex4 = ...

print(litros_ex4)

In [ ]:
# Verificação
assert litros_ex4 is not ... and abs(litros_ex4 - math.pi * 0.6**2 * 1.5 * 1000) <= 1, (
    "Tente de novo, revise: o raio é metade do diâmetro; converta m³ em litros "
    "multiplicando por 1000 (seção 4 - Volume do cilindro)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 5: o rótulo. Uma lata tem raio de 3,3 cm, e o rótulo cobre uma
# faixa de 10 cm de altura em volta dela. Qual é a área do rótulo, em cm²?
area_rotulo_ex5 = ...

print(area_rotulo_ex5)

In [ ]:
# Verificação
assert (
    area_rotulo_ex5 is not ...
    and abs(area_rotulo_ex5 - 2 * math.pi * 3.3 * 10) <= 0.1
), (
    "Tente de novo, revise: o rótulo é a área lateral de um cilindro de raio 3,3 e "
    "altura 10 (seção 3 - Planificação e áreas)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 6: a seção meridiana. A seção meridiana de um cilindro reto
# tem área 48 cm², e a altura do cilindro é 8 cm. Calcule:
# (a) o raio;
# (b) o volume.
raio_ex6 = ...
volume_ex6 = ...

print(raio_ex6, volume_ex6)

In [ ]:
# Verificação
assert raio_ex6 is not ... and abs(raio_ex6 - 3) <= 0.01, (
    "Tente de novo, revise: no item (a), a seção meridiana é um retângulo 2r × h "
    "(seção 2 - Cilindro circular)."
)
assert volume_ex6 is not ... and abs(volume_ex6 - 72 * math.pi) <= 0.01, (
    "Tente de novo, revise: no item (b), com o raio do item (a), o volume é πr²h "
    "(seção 4 - Volume do cilindro)."
)
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: a lata ótima. Para latas de volume 350 cm³, teste os raios de
# 1,50 cm a 8,00 cm, de 0,01 em 0,01, e encontre o raio com a MENOR área total. Guarde a
# razão h / r desse raio em razao_ex7.
melhor_raio = None
menor_area = float("inf")  # qualquer área calculada será menor que "infinito"

for k in range(651):
    raio = 1.5 + k * 0.01
    ...  # complete aqui: calcule a altura e a área total; se a área for a menor até
    # agora, guarde-a em menor_area e guarde o raio em melhor_raio

razao_ex7 = ...  # complete aqui

print(melhor_raio, razao_ex7)

In [ ]:
# Verificação
assert razao_ex7 is not ... and abs(razao_ex7 - 2) <= 0.02, (
    "Tente de novo, revise: para cada raio, calcule h = 350 / (π r²) e a área total; "
    "guarde o raio da menor área (seção 3 - Planificação e áreas)."
)
print("Certo!")
print(
    "O resultado não depende do volume escolhido: para qualquer volume, a lata de "
    "menor área tem h = 2r. Provar isso exige uma ferramenta do Cálculo (a derivada), "
    "que você verá no módulo 11."
)

In [ ]:
# Exercício Desafio 8: a mangueira. Uma mangueira de jardim tem 20 m de comprimento e
# diâmetro interno de 2 cm. Quantos litros de água cabem dentro dela?
litros_ex8 = ...

print(litros_ex8)

In [ ]:
# Verificação
assert litros_ex8 is not ... and abs(litros_ex8 - math.pi * 2000 / 1000) <= 0.01, (
    "Tente de novo, revise: a água dentro da mangueira é um cilindro de raio 1 cm e "
    "altura 2000 cm; 1000 cm³ = 1 L (seção 4 - Volume do cilindro)."
)
print("Certo!")

## Conexões

**Isso depende de:**
- [`0608a_prismas_e_paralelepipedos.ipynb`](0608a_prismas_e_paralelepipedos.ipynb) (prismas inscritos, área lateral)
- [`0609a_volume_e_principio_de_cavalieri.ipynb`](0609a_volume_e_principio_de_cavalieri.ipynb) (Princípio de Cavalieri e volume do prisma)
- [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (comprimento da circunferência) e [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (área do círculo)

**Isso será usado em:**
- 📌 Cone (o cone comparado ao cilindro de mesma base e altura)
- 📌 Esfera (a anticlépsidra, cilindro menos dois cones, e o volume da esfera)
- 📌 Sólidos semelhantes e troncos (tronco de cilindro)
- 📌 Inscrição e circunscrição: corpos redondos (cilindros inscritos e circunscritos)
- 📌 Superfícies e sólidos de revolução (o cilindro gerado por um retângulo girando)
- 📌 Geometria Analítica (a elipse)
- 📌 Limite, Integral e Derivada (o problema da lata ótima)